In [11]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from sklearn.metrics.pairwise import cosine_similarity

# 0) 재현성을 위한 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1) 데이터 로드 및 user_seqs 생성
with open('./util/result_clean.json', 'r') as f:
    raw_data = json.load(f)

rows = []
for uid, user in enumerate(raw_data):
    for ts, token in enumerate(user['token_sequence']):
        rows.append([uid, token, ts])
df = pd.DataFrame(rows, columns=["user_id", "item_id", "timestamp"])
user_seqs = df.groupby("user_id")["item_id"].apply(list).tolist()

# 2) 토큰 ↔ ID 매핑
unique_items = sorted(df["item_id"].unique().tolist())
token2id = {t: i for i, t in enumerate(unique_items)}
id2token = {i: t for t, i in token2id.items()}

n_users = len(user_seqs)
n_items = len(unique_items)

# 3) 사용자 분할: 80% train / 10% val / 10% test
indices = list(range(n_users))
random.shuffle(indices)

n_train = int(0.8 * n_users)
n_val   = int(0.1 * n_users)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

train_seqs = [user_seqs[i] for i in train_idx]
val_seqs   = [user_seqs[i] for i in val_idx]
test_seqs  = [user_seqs[i] for i in test_idx]

# 4) Interaction Matrix 생성 함수
def build_interaction_matrix(seqs, n_items, token2id):
    n_u = len(seqs)
    mat = np.zeros((n_u, n_items), dtype=np.float32)
    for i, seq in enumerate(seqs):
        for item in seq:
            mat[i, token2id[item]] = 1.0
    return mat

train_matrix = build_interaction_matrix(train_seqs, n_items, token2id)
val_matrix   = build_interaction_matrix(val_seqs,   n_items, token2id)

# --- (변경 전) 기존 방식: test_seqs에서 마지막 item 분리 (라벨) ---
test_input_seqs = []
test_labels     = []
for seq in test_seqs:
    if len(seq) < 2:
        test_input_seqs.append([])
        test_labels.append(seq[-1])
    else:
        test_input_seqs.append(seq[:-1])
        test_labels.append(seq[-1])

test_in_matrix = build_interaction_matrix(test_input_seqs, n_items, token2id)

# =========================
# 5) User-based CF
# =========================

# 5.1) Train 사용자들 간 유사도 (사용하지 않아도 괜찮음)
# user_user_sim_train = cosine_similarity(train_matrix)

# 5.2) Test 사용자와 Train 사용자 간 유사도
user_train_sim_test = cosine_similarity(test_in_matrix, train_matrix)
# shape = (n_test, n_train)

# 5.3) 점수 매기기 (가중치 합 방식)
TOP_N = 10
all_scores_user_cf = np.zeros((test_in_matrix.shape[0], n_items), dtype=np.float32)

for test_i in tqdm(range(test_in_matrix.shape[0]), desc="User-based CF scoring"):
    sim_vector    = user_train_sim_test[test_i]            # (n_train,)
    top_neighbors = np.argsort(-sim_vector)[:TOP_N]        # 이웃 상위 TOP_N

    neighbor_interactions = train_matrix[top_neighbors]     # (TOP_N, n_items)
    weights = sim_vector[top_neighbors]                     # (TOP_N,)

    # raw 점수 = ∑(이웃 가중치 × 이웃이 소비한 아이템)
    raw_scores = np.dot(weights, neighbor_interactions)     # (n_items,)

    # 이미 소비한 아이템 제외
    consumed = np.where(test_in_matrix[test_i] > 0.5)[0]
    raw_scores[consumed] = -np.inf

    all_scores_user_cf[test_i] = raw_scores

# =========================
# 6) (변경) Item-based CF
#    → “테스트 유저를 포함하여 매번 아이템-아이템 유사도 계산”
# =========================

all_scores_item_cf = np.zeros((test_in_matrix.shape[0], n_items), dtype=np.float32)

for test_i in tqdm(range(test_in_matrix.shape[0]), desc="Item-based CF scoring"):
    # 6.1) 테스트 유저 벡터
    user_vec = test_in_matrix[test_i]            # shape = (n_items,), 0/1

    # 6.2) Train + Test 유저 하나를 합쳐서 새로운 행렬 구성
    #      → combined_matrix: shape = (n_train + 1, n_items)
    combined_matrix = np.vstack([train_matrix, user_vec[np.newaxis, :]])

    # 6.3) “새로운” 아이템-아이템 유사도 계산
    #      → shape = (n_items, n_items)
    item_item_sim_new = cosine_similarity(combined_matrix.T)

    # 6.4) raw 점수는 test user가 본(1로 표시된) 아이템 i들 ↔ 모든 아이템 j 사이의 유사도 합
    raw_scores = np.dot(user_vec, item_item_sim_new)  # shape = (n_items,)

    # 6.5) 이미 소비한 아이템 제외
    consumed = np.where(user_vec > 0)[0]  # user_vec[i]=1인 인덱스들
    raw_scores[consumed] = -np.inf

    # 6.6) 최종 raw_scores 저장
    all_scores_item_cf[test_i] = raw_scores

# =========================
# 7) Test 라벨을 ID 형태로 변환
# =========================
all_labels = np.array([token2id[tok] for tok in test_labels], dtype=np.int32)

# =========================
# 8) 평가 함수 (질문에서 제시된 그대로)
# =========================
def evaluate_simple_metrics(all_scores: np.ndarray, all_labels: np.ndarray):
    N, V = all_scores.shape
    rank = np.argsort(-all_scores, axis=1)  # shape=(N, V)

    hits1  = np.array([1 if all_labels[i] in rank[i, :1] else 0 for i in range(N)])
    hits5  = np.array([1 if all_labels[i] in rank[i, :5] else 0 for i in range(N)])
    hits10 = np.array([1 if all_labels[i] in rank[i, :10] else 0 for i in range(N)])
    HR1  = hits1.mean()
    HR5  = hits5.mean()
    HR10 = hits10.mean()

    def compute_ndcg_at_k(k):
        dcg_list = np.zeros(N, dtype=np.float32)
        for i in range(N):
            true_item = all_labels[i]
            topk_items = rank[i, :k]
            if true_item in topk_items:
                r = int(np.where(topk_items == true_item)[0][0])
                dcg_list[i] = 1.0 / np.log2(r + 2.0)
            else:
                dcg_list[i] = 0.0
        return float(dcg_list.mean())

    NDCG5  = compute_ndcg_at_k(5)
    NDCG10 = compute_ndcg_at_k(10)

    rr_list = np.zeros(N, dtype=np.float32)
    for i in range(N):
        true_item = all_labels[i]
        r_full = int(np.where(rank[i] == true_item)[0][0])
        rr_list[i] = 1.0 / (r_full + 1.0)
    MRR = float(rr_list.mean())

    return {
        "HR@1":   HR1,
        "HR@5":   HR5,
        "HR@10":  HR10,
        "NDCG@5": NDCG5,
        "NDCG@10":NDCG10,
        "MRR":    MRR
    }

# =========================
# 9) User-based / Item-based 각각 평가
# =========================
metrics_user_cf = evaluate_simple_metrics(all_scores_user_cf, all_labels)
metrics_item_cf = evaluate_simple_metrics(all_scores_item_cf, all_labels)

# 10) 결과 출력
print("=== [Test Set] User-based CF Metrics ===")
for k, v in metrics_user_cf.items():
    print(f"{k}: {v:.4f}")

print("\n=== [Test Set] Item-based CF (재계산) Metrics ===")
for k, v in metrics_item_cf.items():
    print(f"{k}: {v:.4f}")


Item-based CF scoring: 100%|██████████| 601/601 [00:03<00:00, 176.03it/s]


=== [Test Set] User-based CF Metrics ===
HR@1: 0.2047
HR@5: 0.4592
HR@10: 0.5707
NDCG@5: 0.3360
NDCG@10: 0.3724
MRR: 0.3218

=== [Test Set] Item-based CF (재계산) Metrics ===
HR@1: 0.1897
HR@5: 0.3894
HR@10: 0.5258
NDCG@5: 0.2912
NDCG@10: 0.3347
MRR: 0.2959


In [12]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics.pairwise import cosine_similarity

# 0) 재현성을 위한 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1) 데이터 로드 및 user_seqs 생성
with open('./util/result_clean.json', 'r') as f:
    raw_data = json.load(f)

rows = []
for uid, user in enumerate(raw_data):
    for ts, token in enumerate(user['token_sequence']):
        rows.append([uid, token, ts])
df = pd.DataFrame(rows, columns=["user_id", "item_id", "timestamp"])
user_seqs = df.groupby("user_id")["item_id"].apply(list).tolist()

# 2) 토큰 ↔ ID 매핑
unique_items = sorted(df["item_id"].unique().tolist())
token2id = {t: i for i, t in enumerate(unique_items)}
id2token = {i: t for t, i in token2id.items()}

n_users = len(user_seqs)
n_items = len(unique_items)

# 3) 사용자 분할: 80% train / 10% val / 10% test
indices = list(range(n_users))
random.shuffle(indices)

n_train = int(0.8 * n_users)
n_val   = int(0.1 * n_users)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

train_seqs = [user_seqs[i] for i in train_idx]
val_seqs   = [user_seqs[i] for i in val_idx]
test_seqs  = [user_seqs[i] for i in test_idx]

# 4) Interaction Matrix 생성 함수
def build_interaction_matrix(seqs, n_items, token2id):
    n_u = len(seqs)
    mat = np.zeros((n_u, n_items), dtype=np.float32)
    for i, seq in enumerate(seqs):
        for item in seq:
            mat[i, token2id[item]] = 1.0
    return mat

train_matrix = build_interaction_matrix(train_seqs, n_items, token2id)
val_matrix   = build_interaction_matrix(val_seqs,   n_items, token2id)

# --- Test 유저 시퀀스에서 마지막 아이템을 분리하여 “입력 시퀀스”와 “라벨” 생성 ---
test_input_seqs = []
test_labels     = []
for seq in test_seqs:
    if len(seq) < 2:
        # 시퀀스 길이가 1이면, 입력은 빈 리스트, 라벨은 그 하나
        test_input_seqs.append([])
        test_labels.append(seq[-1])
    else:
        # 시퀀스가 2개 이상이면, 마지막 토큰은 라벨, 나머지는 입력
        test_input_seqs.append(seq[:-1])
        test_labels.append(seq[-1])

test_in_matrix = build_interaction_matrix(test_input_seqs, n_items, token2id)

# =========================
# 5) User-based CF (기존 방식)
# =========================

# 5.1) Test 사용자와 Train 사용자 간 유사도 계산
user_train_sim_test = cosine_similarity(test_in_matrix, train_matrix)
# → shape = (n_test, n_train)

# 5.2) Test 사용자별로 점수 매기기 (가중치 합 방식, TOP_N 이웃 사용)
TOP_N = 5  # User-based에서 쓰던 이웃 개수
all_scores_user_cf = np.zeros((test_in_matrix.shape[0], n_items), dtype=np.float32)

for test_i in tqdm(range(test_in_matrix.shape[0]), desc="User-based CF scoring"):
    sim_vector    = user_train_sim_test[test_i]            # (n_train,)
    top_neighbors = np.argsort(-sim_vector)[:TOP_N]        # 가장 유사한 TOP_N train 유저 인덱스

    neighbor_interactions = train_matrix[top_neighbors]     # shape = (TOP_N, n_items)
    weights = sim_vector[top_neighbors]                     # shape = (TOP_N,)

    # user-based raw score = ∑(이웃 가중치 × 이웃의 소비 아이템)
    raw_scores = np.dot(weights, neighbor_interactions)     # shape = (n_items,)

    # 이미 테스트 유저가 입력 시퀀스(과거)에 소비한 아이템은 제외
    consumed = np.where(test_in_matrix[test_i] > 0.5)[0]
    raw_scores[consumed] = -np.inf

    all_scores_user_cf[test_i] = raw_scores

# =========================
# 6) Item-based CF (Top-N neighbor 방식)
#    → “테스트 유저가 본(소비한) 아이템 중에서, 
#       후보 아이템 j와 가장 유사도가 높은 상위 TOP_N개의 아이템을 골라 합산”
# =========================

# 6.1) Train 데이터만으로 아이템-아이템 유사도 한 번 계산
#       → shape = (n_items, n_items)
item_item_sim = cosine_similarity(train_matrix.T)

all_scores_item_cf = np.zeros((test_in_matrix.shape[0], n_items), dtype=np.float32)

for test_i in tqdm(range(test_in_matrix.shape[0]), desc="Item-based CF scoring"):
    user_vec = test_in_matrix[test_i]      # shape = (n_items,), 0/1 (테스트 유저 과거 소비 정보)
    consumed = np.where(user_vec > 0)[0]   # 예: [i1, i2, i3, …]

    raw_scores = np.zeros(n_items, dtype=np.float32)

    for j in range(n_items):
        # 이미 테스트 유저가 소비한 아이템이면 제외
        if j in consumed:
            raw_scores[j] = -np.inf
            continue

        # “j 후보 아이템”과 “테스트 유저가 소비한 모든 아이템 i ∈ consumed” 간 유사도 벡터 추출
        sims = item_item_sim[consumed, j]  # shape = (|consumed|,)

        if len(sims) >= TOP_N:
            # Top-N개의 유사도 값만 골라 합산
            top_n_values = np.sort(sims)[-TOP_N:]
            raw_scores[j] = top_n_values.sum()
        elif len(sims) > 0:
            # 소비한 아이템 개수가 TOP_N 미만이면, 있는 모든 sims를 합산
            raw_scores[j] = sims.sum()
        else:
            # 소비한 아이템이 전혀 없는 경우 → 점수 0
            raw_scores[j] = 0.0

    all_scores_item_cf[test_i] = raw_scores

# =========================
# 7) Test 라벨을 ID 형태로 변환
# =========================
all_labels = np.array([token2id[tok] for tok in test_labels], dtype=np.int32)

# =========================
# 8) 평가 함수 (HR, NDCG, MRR 총 6가지)
# =========================
def evaluate_simple_metrics(all_scores: np.ndarray, all_labels: np.ndarray):
    """
    all_scores: shape=(N, V)  → 각 테스트 유저(i)에 대해 V개 아이템 점수
    all_labels: shape=(N,)    → 각 테스트 유저(i)의 정답 아이템 ID
    """
    N, V = all_scores.shape
    # 1) 점수를 내림차순 정렬한 인덱스(rank) 행렬
    rank = np.argsort(-all_scores, axis=1)  # shape=(N, V)

    # --- HR@1, HR@5, HR@10 계산 ---
    hits1  = np.array([1 if all_labels[i] in rank[i, :1] else 0 for i in range(N)])
    hits5  = np.array([1 if all_labels[i] in rank[i, :5] else 0 for i in range(N)])
    hits10 = np.array([1 if all_labels[i] in rank[i, :10] else 0 for i in range(N)])
    HR1  = hits1.mean()
    HR5  = hits5.mean()
    HR10 = hits10.mean()

    # --- NDCG@5, NDCG@10 계산 (단일 정답이므로 IDCG=1 고정) ---
    def compute_ndcg_at_k(k):
        dcg_list = np.zeros(N, dtype=np.float32)
        for i in range(N):
            true_item = all_labels[i]
            topk_items = rank[i, :k]
            if true_item in topk_items:
                # 0-based index r, DCG = 1 / log2(r+2)
                r = int(np.where(topk_items == true_item)[0][0])
                dcg_list[i] = 1.0 / np.log2(r + 2.0)
            else:
                dcg_list[i] = 0.0
        return float(dcg_list.mean())

    NDCG5  = compute_ndcg_at_k(5)
    NDCG10 = compute_ndcg_at_k(10)

    # --- MRR 계산 ---
    rr_list = np.zeros(N, dtype=np.float32)
    for i in range(N):
        true_item = all_labels[i]
        # 전체 랭킹에서 정답 아이템의 위치 r_full (0-based)
        r_full = int(np.where(rank[i] == true_item)[0][0])
        rr_list[i] = 1.0 / (r_full + 1.0)
    MRR = float(rr_list.mean())

    return {
        "HR@1":   HR1,
        "HR@5":   HR5,
        "HR@10":  HR10,
        "NDCG@5": NDCG5,
        "NDCG@10":NDCG10,
        "MRR":    MRR
    }

# =========================
# 9) User-based / Item-based 각각 평가
# =========================
metrics_user_cf = evaluate_simple_metrics(all_scores_user_cf, all_labels)
metrics_item_cf = evaluate_simple_metrics(all_scores_item_cf, all_labels)

# 10) 결과 출력
print("=== [Test Set] User-based CF Metrics ===")
for k, v in metrics_user_cf.items():
    print(f"{k}: {v:.4f}")

print("\n=== [Test Set] Item-based CF (Top-N 방식) Metrics ===")
for k, v in metrics_item_cf.items():
    print(f"{k}: {v:.4f}")


Item-based CF scoring: 100%|██████████| 601/601 [00:00<00:00, 742.60it/s]


=== [Test Set] User-based CF Metrics ===
HR@1: 0.1814
HR@5: 0.3960
HR@10: 0.4825
NDCG@5: 0.2954
NDCG@10: 0.3238
MRR: 0.2852

=== [Test Set] Item-based CF (Top-N 방식) Metrics ===
HR@1: 0.1614
HR@5: 0.3694
HR@10: 0.4942
NDCG@5: 0.2673
NDCG@10: 0.3075
MRR: 0.2710


In [14]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics.pairwise import cosine_similarity

# 0) 재현성을 위한 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1) 데이터 로드 및 user_seqs 생성
with open('./util/result_clean.json', 'r') as f:
    raw_data = json.load(f)

rows = []
for uid, user in enumerate(raw_data):
    for ts, token in enumerate(user['token_sequence']):
        rows.append([uid, token, ts])
df = pd.DataFrame(rows, columns=["user_id", "item_id", "timestamp"])
user_seqs = df.groupby("user_id")["item_id"].apply(list).tolist()

# 2) 토큰 ↔ ID 매핑
unique_items = sorted(df["item_id"].unique().tolist())
token2id = {t: i for i, t in enumerate(unique_items)}
id2token = {i: t for t, i in token2id.items()}

n_users = len(user_seqs)
n_items = len(unique_items)

# 3) 사용자 분할: 80% train / 10% val / 10% test
indices = list(range(n_users))
random.shuffle(indices)

n_train = int(0.8 * n_users)
n_val   = int(0.1 * n_users)

train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train + n_val]
test_idx  = indices[n_train + n_val:]

train_seqs = [user_seqs[i] for i in train_idx]
val_seqs   = [user_seqs[i] for i in val_idx]
test_seqs  = [user_seqs[i] for i in test_idx]

# 4) Interaction Matrix 생성 함수
def build_interaction_matrix(seqs, n_items, token2id):
    n_u = len(seqs)
    mat = np.zeros((n_u, n_items), dtype=np.float32)
    for i, seq in enumerate(seqs):
        for item in seq:
            mat[i, token2id[item]] = 1.0
    return mat

train_matrix = build_interaction_matrix(train_seqs, n_items, token2id)
val_matrix   = build_interaction_matrix(val_seqs,   n_items, token2id)

# --- Test 유저 시퀀스에서 마지막 아이템을 분리하여 “입력 시퀀스”와 “라벨” 생성 ---
test_input_seqs = []
test_labels     = []
for seq in test_seqs:
    if len(seq) < 2:
        test_input_seqs.append([])
        test_labels.append(seq[-1])
    else:
        test_input_seqs.append(seq[:-1])
        test_labels.append(seq[-1])

test_in_matrix = build_interaction_matrix(test_input_seqs, n_items, token2id)

# =========================
# 5) User-based CF (기존 방식)
# =========================

# 5.1) Test 사용자와 Train 사용자 간 유사도 계산
user_train_sim_test = cosine_similarity(test_in_matrix, train_matrix)
# → shape = (n_test, n_train)

# 5.2) Test 사용자별로 점수 매기기 (가중치 합 방식, TOP_N 이웃 사용)
TOP_N = 10  # 이 값은 User-based와 Item-based에서 동일하게 사용합니다.
all_scores_user_cf = np.zeros((test_in_matrix.shape[0], n_items), dtype=np.float32)

for test_i in range(test_in_matrix.shape[0]):
    sim_vector    = user_train_sim_test[test_i]            # (n_train,)
    top_neighbors = np.argsort(-sim_vector)[:TOP_N]        # 가장 유사한 TOP_N train 유저 인덱스

    neighbor_interactions = train_matrix[top_neighbors]     # shape = (TOP_N, n_items)
    weights = sim_vector[top_neighbors]                     # shape = (TOP_N,)

    raw_scores = np.dot(weights, neighbor_interactions)     # shape = (n_items,)

    consumed = np.where(test_in_matrix[test_i] > 0.5)[0]
    raw_scores[consumed] = -np.inf

    all_scores_user_cf[test_i] = raw_scores

# =========================
# 6) Item-based CF (테스트 유저 포함 → 매번 item–item 유사도 재계산 + TOP_N 적용)
# =========================

all_scores_item_cf = np.zeros((test_in_matrix.shape[0], n_items), dtype=np.float32)

for test_i in range(test_in_matrix.shape[0]):
    user_vec = test_in_matrix[test_i]      # shape = (n_items,), 0/1 (테스트 유저 과거 소비 정보)
    consumed = np.where(user_vec > 0)[0]   # 테스트 유저가 본 아이템 인덱스들

    # 6.1) Train + 해당 테스트 유저를 합친 새로운 사용자–아이템 행렬 생성
    #      → shape = (n_train + 1, n_items)
    combined_matrix = np.vstack([train_matrix, user_vec[np.newaxis, :]])

    # 6.2) “새롭게” 아이템–아이템 유사도 계산
    #      → shape = (n_items, n_items)
    item_item_sim_new = cosine_similarity(combined_matrix.T)

    # 6.3) 각 후보 아이템 j마다, 테스트 유저가 본(consumed) 아이템들과의 유사도 벡터 추출 → 상위 TOP_N개 합산
    raw_scores = np.zeros(n_items, dtype=np.float32)

    for j in range(n_items):
        # 이미 테스트 유저가 소비한 아이템(j)은 추천 대상에서 제외
        if j in consumed:
            raw_scores[j] = -np.inf
            continue

        # “j 후보 아이템”과 “테스트 유저가 소비한 아이템들(consumed)” 간 유사도 벡터
        sims = item_item_sim_new[consumed, j]  # shape = (|consumed|,)

        if len(sims) >= TOP_N:
            # Top-N개의 유사도 값만 골라 합산
            top_n_vals = np.sort(sims)[-TOP_N:]
            raw_scores[j] = top_n_vals.sum()
        elif len(sims) > 0:
            # 만약 소비한 아이템 수 < TOP_N이면, 있는 sims 전부 합산
            raw_scores[j] = sims.sum()
        else:
            # 소비한 아이템이 아예 없는 경우 → 점수 0
            raw_scores[j] = 0.0

    all_scores_item_cf[test_i] = raw_scores

# =========================
# 7) Test 라벨을 ID 형태로 변환
# =========================
all_labels = np.array([token2id[tok] for tok in test_labels], dtype=np.int32)

# =========================
# 8) 평가 함수 (HR, NDCG, MRR 총 6가지)
# =========================
def evaluate_simple_metrics(all_scores: np.ndarray, all_labels: np.ndarray):
    """
    all_scores: shape=(N, V)  → 각 테스트 유저(i)에 대해 V개 아이템 점수
    all_labels: shape=(N,)    → 각 테스트 유저(i)의 정답 아이템 ID
    """
    N, V = all_scores.shape
    # 1) 점수를 내림차순 정렬한 인덱스(rank) 행렬
    rank = np.argsort(-all_scores, axis=1)  # shape=(N, V)

    # --- HR@1, HR@5, HR@10 계산 ---
    hits1  = np.array([1 if all_labels[i] in rank[i, :1] else 0 for i in range(N)])
    hits5  = np.array([1 if all_labels[i] in rank[i, :5] else 0 for i in range(N)])
    hits10 = np.array([1 if all_labels[i] in rank[i, :10] else 0 for i in range(N)])
    HR1  = hits1.mean()
    HR5  = hits5.mean()
    HR10 = hits10.mean()

    # --- NDCG@5, NDCG@10 계산 (단일 정답이므로 IDCG=1 고정) ---
    def compute_ndcg_at_k(k):
        dcg_list = np.zeros(N, dtype=np.float32)
        for i in range(N):
            true_item = all_labels[i]
            topk_items = rank[i, :k]
            if true_item in topk_items:
                # 0-based index r, DCG = 1 / log2(r+2)
                r = int(np.where(topk_items == true_item)[0][0])
                dcg_list[i] = 1.0 / np.log2(r + 2.0)
            else:
                dcg_list[i] = 0.0
        return float(dcg_list.mean())

    NDCG5  = compute_ndcg_at_k(5)
    NDCG10 = compute_ndcg_at_k(10)

    # --- MRR 계산 ---
    rr_list = np.zeros(N, dtype=np.float32)
    for i in range(N):
        true_item = all_labels[i]
        # 전체 랭킹에서 정답 아이템의 위치 r_full (0-based)
        r_full = int(np.where(rank[i] == true_item)[0][0])
        rr_list[i] = 1.0 / (r_full + 1.0)
    MRR = float(rr_list.mean())

    return {
        "HR@1":   HR1,
        "HR@5":   HR5,
        "HR@10":  HR10,
        "NDCG@5": NDCG5,
        "NDCG@10":NDCG10,
        "MRR":    MRR
    }

# =========================
# 9) User-based / Item-based 각각 평가
# =========================
metrics_user_cf = evaluate_simple_metrics(all_scores_user_cf, all_labels)
metrics_item_cf = evaluate_simple_metrics(all_scores_item_cf, all_labels)

# 10) 결과 출력
print("=== [Test Set] User-based CF Metrics ===")
for k, v in metrics_user_cf.items():
    print(f"{k}: {v:.4f}")

print("\n=== [Test Set] Item-based CF (Test 유저 포함, Top-N 방식) Metrics ===")
for k, v in metrics_item_cf.items():
    print(f"{k}: {v:.4f}")


=== [Test Set] User-based CF Metrics ===
HR@1: 0.2047
HR@5: 0.4592
HR@10: 0.5707
NDCG@5: 0.3360
NDCG@10: 0.3724
MRR: 0.3218

=== [Test Set] Item-based CF (Test 유저 포함, Top-N 방식) Metrics ===
HR@1: 0.1897
HR@5: 0.3860
HR@10: 0.5258
NDCG@5: 0.2894
NDCG@10: 0.3340
MRR: 0.2950


In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from sklearn.metrics.pairwise import cosine_similarity

# ─────────────────────────────────────────────────────────────────────────────
# 0) 재현성을 위한 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
# 1) 데이터 로드 및 user_seqs 생성
with open('./util/result_clean.json', 'r') as f:
    raw_data = json.load(f)

rows = []
for uid, user in enumerate(raw_data):
    for ts, token in enumerate(user['token_sequence']):
        rows.append([uid, token, ts])
df = pd.DataFrame(rows, columns=["user_id", "item_id", "timestamp"])
user_seqs = df.groupby("user_id")["item_id"].apply(list).tolist()

# ─────────────────────────────────────────────────────────────────────────────
# 2) 토큰 ↔ ID 매핑
unique_items = sorted(df["item_id"].unique().tolist())
token2id = {t: i for i, t in enumerate(unique_items)}     # 0-based ID
id2token = {i: t for t, i in token2id.items()}

n_users = len(user_seqs)
n_items = len(unique_items)

# ─────────────────────────────────────────────────────────────────────────────
# 3) Leave-one-out 분할: 
#    각 사용자의 시퀀스에서 마지막 아이템은 테스트 라벨, 나머지는 학습 시퀀스
train_seqs = []
test_input_seqs = []
test_labels = []

for seq in user_seqs:
    if len(seq) < 2:
        # 시퀀스 길이가 1 미만이면 학습할 데이터가 부족하므로 skip
        continue
    
    # train_seqs: 마지막 아이템을 제외한 앞부분
    train_seqs.append(seq[:-1])
    
    # test_input_seqs: 마지막 아이템 입력 전까지 (label 제외)
    test_input_seqs.append(seq[:-1])
    
    # test_labels: 마지막 아이템(예측할 정답)
    test_labels.append(seq[-1])

# ─────────────────────────────────────────────────────────────────────────────
# 4) Interaction Matrix 생성 함수
def build_interaction_matrix(seqs, n_items, token2id):
    """
    seqs: list of List[item_token], 각 사용자의 아이템 시퀀스
    n_items: 전체 아이템 개수
    token2id: 아이템 토큰 → ID 매핑 (0-based)
    
    return: numpy array of shape (len(seqs), n_items),
            mat[i, j] = 1이면 i번째 사용자가 j아이템을 과거에 소비함
    """
    n_u = len(seqs)
    mat = np.zeros((n_u, n_items), dtype=np.float32)
    for i, seq in enumerate(seqs):
        for item in seq:
            mat[i, token2id[item]] = 1.0
    return mat

# 4.1) 학습 사용자 행렬 (train_seqs 기반)
train_matrix = build_interaction_matrix(train_seqs, n_items, token2id)  # shape = (n_train_users, n_items)

# 4.2) 테스트 입력 사용자 행렬 (test_input_seqs 기반)
test_in_matrix = build_interaction_matrix(test_input_seqs, n_items, token2id)  # shape = (n_test_users, n_items)

# ─────────────────────────────────────────────────────────────────────────────
# 5) 테스트 라벨을 ID 형태로 변환
all_labels = np.array([token2id[tok] for tok in test_labels], dtype=np.int32)  # shape = (n_test,)

# ─────────────────────────────────────────────────────────────────────────────
# 6) User-based CF: 코사인 유사도를 이용한 KNN 방식
TOP_N = 10  # 이웃 사용자 개수

n_test = test_in_matrix.shape[0]
all_scores_user_cf = np.zeros((n_test, n_items), dtype=np.float32)

# 6.1) 테스트 사용자 vs 학습 사용자 간 유사도 계산
#      shape = (n_test, n_train)
user_train_sim_test = cosine_similarity(test_in_matrix, train_matrix)

for test_i in tqdm(range(n_test)):
    sim_vector    = user_train_sim_test[test_i]         # shape = (n_train,)
    top_neighbors = np.argsort(-sim_vector)[:TOP_N]     # 유사도 상위 TOP_N 인덱스

    neighbor_interactions = train_matrix[top_neighbors]  # shape = (TOP_N, n_items)
    weights = sim_vector[top_neighbors]                  # shape = (TOP_N,)

    # 가중치 합 방식으로 raw score 계산
    raw_scores = np.dot(weights, neighbor_interactions)  # shape = (n_items,)

    # 테스트 유저가 이미 과거에 소비한 아이템 제외 → 점수 -inf 처리
    consumed = np.where(test_in_matrix[test_i] > 0.5)[0]
    raw_scores[consumed] = -np.inf

    all_scores_user_cf[test_i] = raw_scores

# ─────────────────────────────────────────────────────────────────────────────
# 7) Item-based CF: 각 테스트 유저마다 “(train_users + 해당 테스트 유저)” 행렬 합쳐서 
#                    아이템–아이템 유사도를 새로 계산 → KNN 방식
all_scores_item_cf = np.zeros((n_test, n_items), dtype=np.float32)

for test_i in tqdm(range(n_test)):
    user_vec = test_in_matrix[test_i]   # (n_items,), 테스트 유저의 과거 소비 정보 (0/1)
    consumed = np.where(user_vec > 0)[0]  # 테스트 유저가 과거에 소비한 아이템 ID 리스트

    # 7.1) 학습 사용자 행렬 + 이 테스트 유저 벡터를 세로로 합쳐서 새로운 행렬 생성
    #      shape = (n_train + 1, n_items)
    combined_matrix = np.vstack([train_matrix, user_vec[np.newaxis, :]])

    # 7.2) 아이템–아이템 유사도 계산 (n_items x n_items)
    item_item_sim_new = cosine_similarity(combined_matrix.T)  # 각 아이템 간 유사도

    raw_scores = np.zeros(n_items, dtype=np.float32)

    # 7.3) 후보 아이템 j당, 테스트 유저가 소비한 아이템(consumed)과의 유사도 상위 TOP_N 합산
    for j in range(n_items):
        if j in consumed:
            # 이미 소비한 아이템은 추천 대상에서 제외
            raw_scores[j] = -np.inf
            continue

        sims = item_item_sim_new[consumed, j]  # 테스트 유저가 소비한 아이템들 vs 후보 j 간 유사도 벡터

        if len(sims) >= TOP_N:
            top_n_vals = np.sort(sims)[-TOP_N:]
            raw_scores[j] = top_n_vals.sum()
        elif len(sims) > 0:
            raw_scores[j] = sims.sum()
        else:
            # 테스트 유저가 소비한 아이템이 없는 경우
            raw_scores[j] = 0.0

    all_scores_item_cf[test_i] = raw_scores

# ─────────────────────────────────────────────────────────────────────────────
# 8) 평가 함수 정의 (HR, NDCG, MRR)
def evaluate_simple_metrics(all_scores: np.ndarray, all_labels: np.ndarray):
    """
    all_scores: shape=(N, V)  → 각 테스트 유저 i에 대해 V개 아이템 점수
    all_labels: shape=(N,)    → 각 테스트 유저 i의 정답 아이템 ID
    """
    N, V = all_scores.shape
    # 1) 점수를 내림차순 정렬한 인덱스(rank) 행렬
    rank = np.argsort(-all_scores, axis=1)  # shape=(N, V)

    # --- HR@1, HR@5, HR@10 계산 ---
    hits1  = np.array([1 if all_labels[i] in rank[i, :1] else 0 for i in range(N)])
    hits5  = np.array([1 if all_labels[i] in rank[i, :5] else 0 for i in range(N)])
    hits10 = np.array([1 if all_labels[i] in rank[i, :10] else 0 for i in range(N)])
    HR1  = hits1.mean()
    HR5  = hits5.mean()
    HR10 = hits10.mean()

    # --- NDCG@5, NDCG@10 계산 (단일 정답 → IDCG=1 고정) ---
    def compute_ndcg_at_k(k):
        dcg_list = np.zeros(N, dtype=np.float32)
        for i in range(N):
            true_item = all_labels[i]
            topk_items = rank[i, :k]
            if true_item in topk_items:
                r = int(np.where(topk_items == true_item)[0][0])
                dcg_list[i] = 1.0 / np.log2(r + 2.0)  # DCG 공식: 1 / log2(r+2)
            else:
                dcg_list[i] = 0.0
        return float(dcg_list.mean())

    NDCG5  = compute_ndcg_at_k(5)
    NDCG10 = compute_ndcg_at_k(10)

    # --- MRR 계산 ---
    rr_list = np.zeros(N, dtype=np.float32)
    for i in range(N):
        true_item = all_labels[i]
        r_full = int(np.where(rank[i] == true_item)[0][0])  # 전체 랭킹에서 정답 아이템 위치 (0-based)
        rr_list[i] = 1.0 / (r_full + 1.0)
    MRR = float(rr_list.mean())

    return {
        "HR@1":   HR1,
        "HR@5":   HR5,
        "HR@10":  HR10,
        "NDCG@5": NDCG5,
        "NDCG@10":NDCG10,
        "MRR":    MRR
    }

# ─────────────────────────────────────────────────────────────────────────────
# 9) User-based / Item-based 각각 평가
metrics_user_cf = evaluate_simple_metrics(all_scores_user_cf, all_labels)
metrics_item_cf = evaluate_simple_metrics(all_scores_item_cf, all_labels)

# ─────────────────────────────────────────────────────────────────────────────
# 10) 결과 출력
print("=== [Test Set] User-based CF Metrics ===")
for k, v in metrics_user_cf.items():
    print(f"{k}: {v:.4f}")

print("\n=== [Test Set] Item-based CF Metrics ===")
for k, v in metrics_item_cf.items():
    print(f"{k}: {v:.4f}")


100%|██████████| 5993/5993 [00:33<00:00, 179.29it/s]


=== [Test Set] User-based CF Metrics ===
HR@1: 0.0696
HR@5: 0.2696
HR@10: 0.3778
NDCG@5: 0.1720
NDCG@10: 0.2067
MRR: 0.1690

=== [Test Set] Item-based CF Metrics ===
HR@1: 0.0521
HR@5: 0.2481
HR@10: 0.4083
NDCG@5: 0.1510
NDCG@10: 0.2023
MRR: 0.1635
